<a href="https://colab.research.google.com/github/najah24ti-tech/Uji-Validitas-dan-Reabilitas/blob/main/Uji_Validitas_dan_Reabilitas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import re
import pandas as pd
import numpy as np
from scipy import stats
from google.colab import files

In [ ]:
up = files.upload()
nama_file = list(up.keys())[0]
print("File terbaca:", nama_file)

Saving Blank Quiz (Responses).xlsx to Blank Quiz (Responses) (1).xlsx
File terbaca: Blank Quiz (Responses) (1).xlsx


In [ ]:
JUMLAH_KOLOM_DEMOGRAFI = 5 # ubah sesuai jumlah kolom demografi pada data
df = pd.read_excel(nama_file)
butir = df.iloc[:, JUMLAH_KOLOM_DEMOGRAFI:]
print("Jumlah responden :", len(df))
print("Jumlah butir :", butir.shape[1])
print("Data kosong :", int(butir.isna().sum().sum()))

Jumlah responden : 38
Jumlah butir : 31
Data kosong : 0


In [ ]:
skala = {
"Sangat Tidak Setuju (STS)": 1,
"Tidak Setuju (TS)": 2,
"Netral (N)": 3,
"Setuju (S)": 4,
"Sangat Setuju (SS)": 5,
}
num = butir.replace(skala).apply(pd.to_numeric, errors="coerce")
# Harus 0. Jika lebih dari 0, ada jawaban yang tidak terbaca.
print("Sel yang gagal dikonversi:", int(num.isna().sum().sum()))
num.head()

Sel yang gagal dikonversi: 43


/tmp/ipykernel_553/2330314235.py:8: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  num = butir.replace(skala).apply(pd.to_numeric, errors="coerce")


,Jumlah Platform AI yang berlangganan secara premium (bukan free),Penggunaan AI pada tugas atau projek kuliah [Saya menggunakan platform AI hampir setiap kali mendapatkan tugas akademik.],Penggunaan AI pada tugas atau projek kuliah [Saya menggunakan AI untuk mencari solusi atas permasalahan atau soal perkuliahan.],Penggunaan AI pada tugas atau projek kuliah [Saya menggunakan AI untuk mencari ide atau inspirasi dalam mengerjakan tugas./projek],Penggunaan AI pada tugas atau projek kuliah [Saya merasa kesulitan memulai mengerjakan tugas tanpa bantuan AI.],Penggunaan AI pada tugas atau projek kuliah [Saya merasa kurang yakin dengan hasil tugas apabila tidak menggunakan AI.],Penggunaan AI pada tugas atau projek kuliah [Saya percaya bahwa AI dapat memberikan informasi yang membantu penyelesaian tugas.],Penggunaan AI pada tugas atau projek kuliah [Saya memeriksa kembali kebenaran informasi yang diberikan AI sebelum menggunakannya.],Aspek Psikologis penggunaan AI pada tugas atau projek kuliah [Saya merasa perlu menggunakan AI setiap kali mendapatkan tugas atau proyek kuliah.],Aspek Psikologis penggunaan AI pada tugas atau projek kuliah [aya merasa kesulitan menyelesaikan tugas kuliah apabila tidak menggunakan bantuan AI.],...,Kebiasaan penggunaan teknologi digital [Saya jarang memeriksa kembali hasil dari AI karena sudah percaya terhadap ketepatan jawabannya.],Prokrastinasi akademik. [Saya cenderung menunggu hingga mendekati deadline untuk mulai mengerjakan tugas.],Prokrastinasi akademik. [Saya sering menunda pengerjaan tugas karena merasa dapat dikerjakan dengan mudah dan cepat dengan menggunakan AI],Prokrastinasi akademik. [Saya sering menyelesaikan tugas kuliah dalam waktu yang sangat dekat dengan batas pengumpulan.],Prokrastinasi akademik. [Saya sering mengerjakan beberapa tugas sekaligus karena sebelumnya menunda pengerjaannya.],Rsspon emosional [Saya merasa tertekan ketika harus menyelesaikan tugas dalam waktu singkat akibat menunda pengerjaannya.],Rsspon emosional [Saya merasa panik ketika beberapa tugas yang saya tunda memiliki batas waktu pengumpulan yang berdekatan.],Rsspon emosional [Saya merasa kurang puas terhadap hasil tugas yang dikerjakan karena terburu-buru mendekati deadline.],Rsspon emosional [Saya merasa lega setelah berhasil mengumpulkan tugas meskipun pengerjaannya dilakukan AI saat mendekati deadline.],Rsspon emosional [Saya merasa menyesal karena hasil tugas mungkin bisa lebih baik apabila dikerjakan lebih awal.]
0,2.0,3,4,4,4,3,3,4,3,4,...,3,3,3,3,4,4,3,4,4,3
1,1.0,4,4,4,2,3,4,5,3,2,...,2,3,3,3,4,4,4,4,3,4
2,1.0,3,4,5,3,4,4,5,4,4,...,2,3,3,3,3,4,4,3,4,4
3,1.0,5,5,5,5,4,5,5,5,4,...,5,5,4,4,5,4,5,4,5,5
4,1.0,5,4,4,3,4,4,4,3,4,...,2,4,4,3,4,3,4,5,4,5


In [ ]:
dimensi = {}
for kolom in butir.columns:
    hasil = re.match(r"^\s*(.*?)\s*\[", str(kolom))
    if hasil:
        nama = hasil.group(1).strip().rstrip(".")
        if nama == "Rsspon emosional":
            nama = "Respons emosional"
        dimensi.setdefault(nama, []).append(kolom)
for nama, daftar in dimensi.items():
    print(f"{nama}: {len(daftar)} butir")

Penggunaan AI pada tugas atau projek kuliah: 7 butir
Aspek Psikologis penggunaan AI pada tugas atau projek kuliah: 7 butir
Kebiasaan penggunaan teknologi digital: 6 butir
Prokrastinasi akademik: 4 butir
Respons emosional: 5 butir


In [ ]:
n = len(num)
t = stats.t.ppf(0.975, n - 2)
r_tabel = t / np.sqrt((n - 2) + t**2)
print(f"n = {n} | r tabel (α=0,05; df={n-2}) = {r_tabel:.3f}")
hasil_validitas = []
for nama, daftar in dimensi.items():
    x = num[daftar]
    total = x.sum(axis=1)
    for i, kolom in enumerate(daftar, 1):
        r, p = stats.pearsonr(x[kolom], total)
        r_koreksi, _ = stats.pearsonr(x[kolom], total - x[kolom])
        hasil_validitas.append({
            "Dimensi": nama,
            "Butir": f"B{i}",
            "r hitung": round(r, 3),
            "Corrected r": round(r_koreksi, 3),
            "p-value": round(p, 4),
            "Keterangan": "Valid" if r > r_tabel else "Tidak valid",
        })
tabel_validitas = pd.DataFrame(hasil_validitas)
tabel_validitas

n = 38 | r tabel (α=0,05; df=36) = 0.320


,Dimensi,Butir,r hitung,Corrected r,p-value,Keterangan
0,Penggunaan AI pada tugas atau projek kuliah,B1,0.795,0.705,0.0000,Valid
1,Penggunaan AI pada tugas atau projek kuliah,B2,0.744,0.641,0.0000,Valid
2,Penggunaan AI pada tugas atau projek kuliah,B3,0.797,0.713,0.0000,Valid
3,Penggunaan AI pada tugas atau projek kuliah,B4,0.464,0.330,0.0033,Valid
4,Penggunaan AI pada tugas atau projek kuliah,B5,0.772,0.662,0.0000,Valid
5,Penggunaan AI pada tugas atau projek kuliah,B6,0.838,0.756,0.0000,Valid
6,Penggunaan AI pada tugas atau projek kuliah,B7,0.757,0.644,0.0000,Valid
7,Aspek Psikologis penggunaan AI pada tugas atau...,B1,0.861,0.800,0.0000,Valid
8,Aspek Psikologis penggunaan AI pada tugas atau...,B2,0.793,0.715,0.0000,Valid
9,Aspek Psikologis penggunaan AI pada tugas atau...,B3,0.763,0.662,0.0000,Valid


In [ ]:
def cronbach_alpha(x):
    k = x.shape[1]
    return k / (k - 1) * (
        1 - x.var(ddof=1).sum() / x.sum(axis=1).var(ddof=1)
    )
def kategori(a):
    return (
        "Sangat baik" if a >= 0.90
        else "Baik" if a >= 0.80
        else "Reliabel" if a >= 0.70
        else "Tidak reliabel"
    )
hasil_reliabilitas = []
alpha_dihapus = []
for nama, daftar in dimensi.items():
    x = num[daftar]
    a = cronbach_alpha(x)
    hasil_reliabilitas.append({
        "Dimensi": nama,
        "Jumlah butir": len(daftar),
        "Cronbach's alpha": round(a, 3),
        "Kategori": kategori(a),
    })
    for i, kolom in enumerate(daftar, 1):
        if len(daftar) > 2:
            alpha_dihapus.append({
                "Dimensi": nama,
                "Butir": f"B{i}",
                "Alpha jika dihapus": round(
                    cronbach_alpha(x.drop(columns=kolom)), 3
                ),
            })
tabel_reliabilitas = pd.DataFrame(hasil_reliabilitas)
tabel_alpha_dihapus = pd.DataFrame(alpha_dihapus)
display(tabel_reliabilitas)
display(tabel_alpha_dihapus)

,Dimensi,Jumlah butir,Cronbach's alpha,Kategori
0,Penggunaan AI pada tugas atau projek kuliah,7,0.865,Baik
1,Aspek Psikologis penggunaan AI pada tugas atau...,7,0.904,Sangat baik
2,Kebiasaan penggunaan teknologi digital,6,0.777,Reliabel
3,Prokrastinasi akademik,4,0.851,Baik
4,Respons emosional,5,0.873,Baik


,Dimensi,Butir,Alpha jika dihapus
0,Penggunaan AI pada tugas atau projek kuliah,B1,0.837
1,Penggunaan AI pada tugas atau projek kuliah,B2,0.846
2,Penggunaan AI pada tugas atau projek kuliah,B3,0.836
3,Penggunaan AI pada tugas atau projek kuliah,B4,0.881
4,Penggunaan AI pada tugas atau projek kuliah,B5,0.843
5,Penggunaan AI pada tugas atau projek kuliah,B6,0.829
6,Penggunaan AI pada tugas atau projek kuliah,B7,0.846
7,Aspek Psikologis penggunaan AI pada tugas atau...,B1,0.879
8,Aspek Psikologis penggunaan AI pada tugas atau...,B2,0.889
9,Aspek Psikologis penggunaan AI pada tugas atau...,B3,0.896


In [ ]:
hasil_perbandingan = []

for nama, grup in tabel_validitas.groupby("Dimensi"):
    hasil_perbandingan.append({
        "Dimensi": nama,
        "Valid berdasarkan r hitung": (grup["Keterangan"] == "Valid").sum(),
        "Valid berdasarkan Corrected r": (grup["Corrected r"] >= 0.30).sum(),
        "Tidak valid berdasarkan Corrected r": (grup["Corrected r"] < 0.30).sum(),
    })

tabel_perbandingan = pd.DataFrame(hasil_perbandingan)

display(tabel_perbandingan)

,Dimensi,Valid berdasarkan r hitung,Valid berdasarkan Corrected r,Tidak valid berdasarkan Corrected r
0,Aspek Psikologis penggunaan AI pada tugas atau...,7,7,0
1,Kebiasaan penggunaan teknologi digital,6,6,0
2,Penggunaan AI pada tugas atau projek kuliah,7,7,0
3,Prokrastinasi akademik,4,4,0
4,Respons emosional,5,5,0


In [ ]:
alpha_awal = tabel_reliabilitas[
    ["Dimensi", "Cronbach's alpha"]
].rename(columns={
    "Cronbach's alpha": "Alpha awal"
})

tabel_evaluasi = tabel_alpha_dihapus.merge(
    alpha_awal,
    on="Dimensi",
    how="left"
)

tabel_evaluasi["Keputusan"] = tabel_evaluasi.apply(
    lambda baris: "Perlu ditinjau"
    if baris["Alpha jika dihapus"] > baris["Alpha awal"]
    else "Dipertahankan",
    axis=1
)

display(tabel_evaluasi)

,Dimensi,Butir,Alpha jika dihapus,Alpha awal,Keputusan
0,Penggunaan AI pada tugas atau projek kuliah,B1,0.837,0.865,Dipertahankan
1,Penggunaan AI pada tugas atau projek kuliah,B2,0.846,0.865,Dipertahankan
2,Penggunaan AI pada tugas atau projek kuliah,B3,0.836,0.865,Dipertahankan
3,Penggunaan AI pada tugas atau projek kuliah,B4,0.881,0.865,Perlu ditinjau
4,Penggunaan AI pada tugas atau projek kuliah,B5,0.843,0.865,Dipertahankan
5,Penggunaan AI pada tugas atau projek kuliah,B6,0.829,0.865,Dipertahankan
6,Penggunaan AI pada tugas atau projek kuliah,B7,0.846,0.865,Dipertahankan
7,Aspek Psikologis penggunaan AI pada tugas atau...,B1,0.879,0.904,Dipertahankan
8,Aspek Psikologis penggunaan AI pada tugas atau...,B2,0.889,0.904,Dipertahankan
9,Aspek Psikologis penggunaan AI pada tugas atau...,B3,0.896,0.904,Dipertahankan


In [ ]:
print("RINGKASAN HASIL PENGUJIAN")
print("==========================")
print("Jumlah responden :", len(num))
print("Jumlah dimensi   :", len(dimensi))
print("Jumlah butir     :", num.shape[1])

print("\nHasil Validitas:")
print("Butir valid     :", (tabel_validitas["Keterangan"] == "Valid").sum())
print("Butir tidak valid:", (tabel_validitas["Keterangan"] == "Tidak valid").sum())

print("\nHasil Reliabilitas:")
print("Dimensi reliabel:", (tabel_reliabilitas["Cronbach's alpha"] >= 0.70).sum())
print("Dimensi tidak reliabel:", (tabel_reliabilitas["Cronbach's alpha"] < 0.70).sum())

RINGKASAN HASIL PENGUJIAN
Jumlah responden : 38
Jumlah dimensi   : 5
Jumlah butir     : 31

Hasil Validitas:
Butir valid     : 29
Butir tidak valid: 0

Hasil Reliabilitas:
Dimensi reliabel: 5
Dimensi tidak reliabel: 0


In [ ]:
gabung = tabel_validitas.merge(
    tabel_alpha_dihapus,
    on=["Dimensi", "Butir"],
    how="left"
)

with pd.ExcelWriter("hasil_uji_validitas_reliabilitas.xlsx") as w:
    gabung.to_excel(w, sheet_name="Validitas", index=False)
    tabel_reliabilitas.to_excel(w, sheet_name="Reliabilitas", index=False)

files.download("hasil_uji_validitas_reliabilitas.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>